In [251]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv('../data/Wildlife_Export_2023_2025.csv', encoding='latin-1')

### Stripping the Column Headers

In [ ]:
df.columns = df.columns.str.strip()

### Getting More Information About the Data
* There are 65,774 records for the time period between 2023 & 2025
* This dataset has 102 columns, much of this data can probably be dropped.

In [ ]:
df.shape

### Checking the datatype for all columns
<u>Issues with the datatypes</u>
* `INCIDENT_DATE` is a string
* `TIME` is a string
* `NR_INJURIES` is a float
* `NR_FATALITIES` is a float

In [ ]:
df.info(verbose=True, show_counts=True)

### Checking for Null Data
- `null_pct` calculates the percentage of missing values for each column.
- The print statement filters to columns where more than **50%** of values are null.
- Columns exceeding this threshold will be candidates for removal.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct[null_pct > 50])

### Converting `INCIDENT_DATE` to `datetime`
* This code will convert `INCIDENT_DATE` to `datetime` which will provide for more robust data. 

In [ ]:
df['INCIDENT_DATE'] = pd.to_datetime(df['INCIDENT_DATE'])

In [258]:
df.shape

(46503, 75)

### Comparing TIME and TIME_OF_DAY
- `TIME` reports zero nulls, while `TIME_OF_DAY` is missing 47% of its values.
    - However, inspecting the top 20 rows via `head(20)` reveals visible blanks in `TIME` — suggesting these are empty strings rather than true nulls, since the column's `str` dtype won't flag them with `isna()`.

In [259]:
df['TIME'].head(20)

0       NaN
1     13:49
2      7:15
10    10:17
11    12:30
12      NaN
13    14:37
14     0:22
15     8:12
16      NaN
17      NaN
18      NaN
19      NaN
22    21:10
27    14:51
28    22:25
29    19:31
30      NaN
31      NaN
32      NaN
Name: TIME, dtype: str

In [260]:
print(df['TIME'].isna().sum(), df['TIME'].isna().mean())
print(df['TIME'].isnull().sum(), df['TIME'].isnull().mean())
print(df['TIME_OF_DAY'].isna().sum(), df['TIME_OF_DAY'].isna().mean())

8945 0.19235318151517106
8945 0.19235318151517106
12546 0.2697890458680085


When I query the `TIME` column for empty strings, it returns `0` - suggesting that more work needs to be done to this column.

In [299]:
print((df['TIME'] == '').sum())

0


This code strips whitespace from the `TIME` column and replaces any blank values with `NaN`. Querying `TIME` for `NaN` returns 21,865 records, confirming that a significant portion of the dataset is missing time values.

In [ ]:
df['TIME'] = df['TIME'].str.strip().replace('', np.nan)

In [298]:
print(df['TIME'].isna().sum())

28216


This code does the same of `TIME_OF_DAY`, resulting in 31,410 values missing in the dataset for time of day.

In [264]:
print(df_AC['TIME_OF_DAY'].isna().sum())

12546


### Exploration of `TIME` and `TIME_OF_DAY`
There appears to be an inverse relationship between `TIME` and `TIME_OF_DAY`: when one is populated, the other may be missing. To reconcile these columns, I'll define a `time_to_minutes()` function that splits the `TIME` string on the colon and converts it to total elapsed minutes (hours × 60 + minutes). From there, I can derive a unified `TIME_OF_DAY` column that captures the time of day for every event.
- [ ] Confirm if this inverse relationship exists
- [ ] Determin if `TIME` can inform `TIME_OF_DAY`

In [295]:
def time_to_minutes(t):
    if pd.isna(t) or t == '':
        return np.nan
    h, m = t.split(':')
    return int(h) * 60 + int(m)

df['TIME_MINUTES'] = df['TIME'].apply(time_to_minutes)

In [296]:
df[['TIME_MINUTES','TIME_OF_DAY']].head(20)

,TIME_MINUTES,TIME_OF_DAY
0,NaN,Night
1,829.0,Day
2,435.0,NaN
3,850.0,NaN
4,530.0,NaN
5,612.0,NaN
6,630.0,NaN
7,530.0,NaN
8,630.0,NaN
9,1230.0,NaN


### Considering and Adjusting `NR_INJURIES` and `NR_FATALITIES`
* `NR_INJURIES` and `NR_FATALITIES` are both floats. These should be converted to ints because a person is either injured or not.
* `NR_INJURIES` is 99.96% empty and `NR_FATALITIES` is 99.99% empty, but the data that does exist is still interesting enough to keep at this time.

In [268]:
df['NR_INJURIES'] = df['NR_INJURIES'].fillna(0).astype(int)

In [269]:
df['NR_FATALITIES'] = df['NR_FATALITIES'].fillna(0).astype(int)

### Checking for Duplicates
* There are no duplicate rows

In [270]:
df.duplicated().sum()

np.int64(0)

### Initial Observations About the Data
* `INCIDENT_YEAR` The number of bird strikes is increasing year-over-year from 2023 to 2025.
* `INCIDENT_MONTH` Most bird strikes are recorded in September, August, October and July. There seems to be a significant difference between the warmer and colder months.
* `STATE` The top 5 states are TX, FL, CA, CO and TN. This seems to track with where the busy airports are (perhaps I can bring some data in regarding airport activity). Kentuky made the top 10.
* `FAAREGION` There seems to be some duplication in the FAAREGION data. I will need to strip this column to normalize it.
* `AIRPORT` UKNOWN is by far the largest airport, I am curious why that is the case. My guess is that it because many of the bird strikes are from observation of evidence of a bird strike but a lack of clarity where it happened. The other airports make sense because they are large market hubs.
* `OPERATOR` Again, UNKNOWN is by far the largest operator followed by the other popular airlines. I do find it interesting that business beats Delta and United, as those are very popular airlines.
* `PHASE_OF_FLIGHT` This is interesting because it describes at what point a plane is most likely to have a bird strike. I will need to supliment my understanding of these phases of flight because I do not know what makes them distinct.




In [ ]:
df['INCIDENT_YEAR'].value_counts()

In [ ]:
df['INCIDENT_MONTH'].value_counts()

In [ ]:
df['STATE'].value_counts().head(10)

In [ ]:
df['FAAREGION'].value_counts().head(10)

In [ ]:
df['AIRPORT'].value_counts().head(10)

In [ ]:
df['OPERATOR'].value_counts().head(10)

In [ ]:
df['NR_FATALITIES'].value_counts()

In [ ]:
df['PHASE_OF_FLIGHT'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].isna().sum()

In [ ]:
df['TIME_OF_DAY'].value_counts(dropna=False)

In [ ]:
df['TIME'].head(10)

In [272]:
df['INGESTED_OTHER'].value_counts()

INGESTED_OTHER
False    65724
True        50
Name: count, dtype: int64

In [274]:
df['ENG_1_POS'].value_counts()

ENG_1_POS
1.0    34845
5.0     6521
7.0     2853
6.0     1062
4.0     1018
3.0       68
2.0       25
Name: count, dtype: int64

### Exploring the Danger of Bird Strikes
The original dataset has two fields to display number of injuries and deaths due to FAA bird strikes. I will create a new dataframe to just include columns that add context to the strikes that caused injury or death called `df_human_impact`.
* On January 1, 2024, there was an incident where 3 individuals died after striking a Cackling Goose.
* There are 22 injuries recorded between spanning between 2023-2025, one of them being a white-tailed deer.


In [ ]:
df_human_impact = df[['INCIDENT_DATE','AIRCRAFT','NR_INJURIES','NR_FATALITIES','SPECIES','ENROUTE_STATE']]

In [ ]:
df_human_impact[df_human_impact['NR_FATALITIES']>0]

In [ ]:
df_human_impact[df_human_impact['NR_INJURIES']>0].sort_values('NR_INJURIES', ascending=False)

### Exploration of Birds
**Most Frequently Observed Species** "Unknown bird" and "Unknown bird - small" are the two most commonly recorded strike species. This likely reflects the difficulty of identification mid-air or the severity of the collision leaving little physical evidence. The remaining species are common birds found throughout the country.

`BIRD_BAND_NUMBER` This column initially seemed promising for cross-referencing external data, but reliable additional resources are limited. This column will likely be dropped from further analysis.

` NUM_STRUCK` - **Surprising Finding** The frequency of multi-bird strikes (11-100 and 100+) was unexpected. This challenges the assumption that this would not happen often.

`SIZE` Strike distribution by size follows an expected pattern: small birds are most frquently struck, followed by medium, then large. 

In [275]:
df_AC['SPECIES'].value_counts().head(10)

SPECIES
Unknown bird                 14281
Unknown bird - small          7010
Barn swallow                  1534
Mourning dove                 1441
Unknown bird - medium         1371
Horned lark                   1193
American kestrel               682
Killdeer                       669
Cliff swallow                  610
Brazilian free-tailed bat      507
Name: count, dtype: int64

In [277]:
df['BIRD_BAND_NUMBER'].value_counts().head()

BIRD_BAND_NUMBER
BANDED       23
225730859     1
196701236     1
194737222     1
194754132     1
Name: count, dtype: int64

In [278]:
df_AC['NUM_STRUCK'].value_counts()

NUM_STRUCK
1                 42332
 2-10              3895
 11-100             157
                    114
 More than 100        5
Name: count, dtype: int64

In [279]:
df_AC['SIZE'].value_counts()

SIZE
Small     26804
Medium     3628
Large      1738
Name: count, dtype: int64

### Species and Size: Grouping BIrds for Analysis
Segmenting birds by size category (small, medium, large) opens up several useful lines of questions:

* **Most common species by size** - Which bird species is most frequently involved in strikes within each size category?
* **Strike frequency by size** - Are certain size categories struck more often than others?
* **Impact severity by size** - Does bird size correlate with the degree of damage or operational effect on the aircraft?
* **Size vs. flight phase** - Are birds of different sizes more likely to be struck during specific phases of flight?

In [280]:
df_LGBIRD = df_AC[df_AC['SIZE'] == 'Large']

In [281]:
df_MDBIRD = df_AC[df_AC['SIZE'] == 'Medium']

In [282]:
df_SMBIRD = df_AC[df_AC['SIZE'] == 'Small']

### Top 10 Large, Medium, and Small Birds
#### Explanation
Using `value_counts().head(10)` to identify the top 10 species involved in strikes within each size category.

#### Observations
- **Large:** The category includes non-bird wildlife — skunks, coyotes, jackrabbits, and deer — alongside the top entry, `Unknown bird - large`. The presence of ground animals is unexpected given typical airport perimeter fencing, suggesting these strikes may be concentrated at smaller, less-secured airports.
- **Medium:** Predominantly birds, with `Unknown bird - medium` as the most frequent entry.
- **Small:** Same pattern as medium — `Unknown bird - small` leads the category.

In [287]:
df_LGBIRD['SPECIES'].value_counts().head(10)

SPECIES
Unknown bird - large    353
Turkey vulture          182
Canada goose            173
Bald eagle              113
Coyote                  113
White-tailed deer       105
Black vulture            93
Osprey                   93
Great blue heron         57
Geese                    32
Name: count, dtype: int64

In [284]:
df_MDBIRD['SPECIES'].value_counts().head(10)

SPECIES
Unknown bird - medium        1371
Red-tailed hawk               323
Gulls                         284
Ring-billed gull              161
Mallard                       134
White-headed gull complex     132
Herring gull                  112
American barn owl              98
Hawks                          96
American coot                  54
Name: count, dtype: int64

In [285]:
df_SMBIRD['SPECIES'].value_counts().head(10)

SPECIES
Unknown bird - small         7010
Barn swallow                 1534
Mourning dove                1441
Horned lark                  1193
American kestrel              682
Killdeer                      669
Cliff swallow                 610
Brazilian free-tailed bat     507
European starling             495
American robin                486
Name: count, dtype: int64

### Exploration of Airplane and Flight Data
* `AC_MASS` is a float ranging from 1 - 5

In [306]:
df['AC_MASS'].value_counts()

AC_MASS
4.0    37939
3.0     3017
1.0     2846
2.0     2129
5.0      572
Name: count, dtype: int64

In [305]:
df['AC_MASS'].describe()

count    46503.000000
mean         3.672258
std          0.838874
min          1.000000
25%          4.000000
50%          4.000000
75%          4.000000
max          5.000000
Name: AC_MASS, dtype: float64

In [304]:
df['AIRCRAFT'].value_counts()

AIRCRAFT
UNKNOWN       19160
EMB-170        5152
B-737-800      4853
B-737-8        4034
A-320          3763
              ...  
GROB 120A         1
F-15E             1
AT-502            1
G400              1
B737-Max 9        1
Name: count, Length: 397, dtype: int64

In [303]:
df['HEIGHT'].describe()

count    26486.000000
mean       961.765121
std       1911.914579
min          0.000000
25%          0.000000
50%         50.000000
75%       1000.000000
max      32000.000000
Name: HEIGHT, dtype: float64

In [294]:
bin_height = [0, 50, 500, 1000, 5000, 32000]

labels = ['Ground (0-50ft)', 'Low (51-500ft)', 'Pattern (501-1000ft)', 'Approach (1001-5000ft)', 'Cruise (5001ft+)']

df_AC['HEIGHT_BIN'] = pd.cut(df['HEIGHT'], bins=bin_height, labels=labels, include_lowest=True)

print(df_AC['HEIGHT_BIN'].value_counts())

HEIGHT_BIN
Ground (0-50ft)           13704
Approach (1001-5000ft)     5307
Low (51-500ft)             4099
Pattern (501-1000ft)       2132
Cruise (5001ft+)           1014
Name: count, dtype: int64
